<a href="https://colab.research.google.com/github/Preetitamrakar-phd/GenAI_Hands-on/blob/main/RAG_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Complete RAG Pipeline
# 1. Complete RAG architecture: retrieve → inject → generate
# 2. LangChain components (retrievers, prompts, chains)
# 3. Anti-hallucination strategies
# 4. Building a production-ready Q&A system

In [2]:
#!pip install langchain_chroma, langchain_openai, langchain_text_splitters


In [6]:
import json
import os
import time
from operator import itemgetter  # Used in exercises/solutions for LCEL key extraction
# LangChain is a framework for building LLM applications
from langchain_openai import OpenAIEmbeddings, ChatOpenAI  # OpenAI integrations
from langchain_chroma import Chroma  # Vector database for similarity search
from langchain_text_splitters import RecursiveCharacterTextSplitter  # Smart text chunking
from langchain_core.documents import Document  # Document abstraction
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder  # Prompt templates
from langchain_core.messages import HumanMessage, AIMessage  # Chat history message types
from langchain_core.output_parsers import StrOutputParser  # Parse LLM output
from langchain_core.runnables import RunnablePassthrough  # Pass data through pipeline

In [4]:
from google.colab import files
uploaded = files.upload()

Saving synthetic_tickets.json to synthetic_tickets.json


In [5]:
# Load environment variables (API keys, model names)
from dotenv import load_dotenv
load_dotenv()

True

PART 1: Data Ingestion & Vector Store Setup

In [7]:
# Load tickets
with open('synthetic_tickets.json', 'r') as f:
    tickets = json.load(f)
print(f"✓ Loaded {len(tickets)} support tickets")

✓ Loaded 20 support tickets


In [8]:
# Convert to LangChain Document objects
# Documents are the core abstraction in LangChain - they combine content with metadata

documents = []
for ticket in tickets:
    # Create rich document with all context
    # TIP: Structure your content logically - LLMs understand formatted text better
    content = f"""
Ticket ID: {ticket['ticket_id']}
Title: {ticket['title']}
Category: {ticket['category']}
Priority: {ticket['priority']}
Date: {ticket['created_date']} to {ticket['resolved_date']}

Problem Description:
{ticket['description']}

Resolution:
{ticket['resolution']}
    """.strip()

    # Create Document with metadata
    # Metadata is crucial for filtering, citation, and source tracking
    # Best practice: Include all information you might want to filter or display later
    doc = Document(
        page_content=content,  # The actual text content
        metadata={  # Structured data about the document
            'ticket_id': ticket['ticket_id'],
            'title': ticket['title'],
            'category': ticket['category'],
            'priority': ticket['priority'],
            'source': f"Ticket {ticket['ticket_id']}"
        }
    )
    documents.append(doc)

print(f"✓ Created {len(documents)} documents with metadata")

✓ Created 20 documents with metadata


In [9]:
print("\nInitializing OpenAI embedding model...")
embeddings = OpenAIEmbeddings(
    model=os.getenv('OPENAI_EMBEDDING_MODEL', 'text-embedding-3-small')
)
print("✓ OpenAI embedding model ready")


Initializing OpenAI embedding model...
✓ OpenAI embedding model ready


In [10]:
# Build vector store using Chroma
# Chroma is an open-source vector database optimized for AI applications
# It stores embeddings and enables fast similarity search
print("\nBuilding Chroma vector store...")
# Always rebuild from scratch so repeated runs don't accumulate duplicate documents.


Building Chroma vector store...


In [11]:
import shutil
persist_directory = "./rag_vectorstore"
if os.path.exists(persist_directory):
    try:
        shutil.rmtree(persist_directory)
    except PermissionError:
        # Windows may keep Chroma files locked briefly from a prior run.
        # Fall back to a unique directory so the demo can continue.
        persist_directory = f"./rag_vectorstore_{int(time.time())}"
        print(f"⚠ Vector store directory is locked; using {persist_directory} instead")
vector_store = Chroma.from_documents(
    documents=documents,  # Our support ticket documents
    embedding=embeddings,  # Embedding function to use
    collection_name="supportdesk_rag",  # Name for this collection
    persist_directory=persist_directory,  # Where to save the database
    collection_metadata={"hnsw:space": "cosine"}  # Use cosine distance for similarity search
)
print("✓ Vector store created and persisted")

✓ Vector store created and persisted


PART 2: Create Retriever

In [12]:
# Create a retriever from the vector store
# Retrievers are the interface for querying the vector store
retriever = vector_store.as_retriever(
    search_type="similarity",  # Use cosine similarity for ranking
    search_kwargs={"k": 3}  # Retrieve top-3 most similar documents
    # Other options:
    # - "mmr" (Maximal Marginal Relevance): Balances relevance with diversity
    # - "similarity_score_threshold": Only return docs above a score threshold
)

In [13]:
print("✓ Retriever configured:")
print("\nTIP: k=3-5 is usually optimal. Too few → missing context, too many → noise")

✓ Retriever configured:

TIP: k=3-5 is usually optimal. Too few → missing context, too many → noise


In [14]:
# Test retriever
test_query = "Users can't log in after changing passwords"
print(f"\nTest query: '{test_query}'")
retrieved_docs = retriever.invoke(test_query)

print(f"\nRetrieved {len(retrieved_docs)} documents:")
for i, doc in enumerate(retrieved_docs, 1):
    print(f"\n#{i} - {doc.metadata['ticket_id']}: {doc.metadata['title']}")
    print(f"  Category: {doc.metadata['category']}")


Test query: 'Users can't log in after changing passwords'

Retrieved 3 documents:

#1 - TICK-001: Users unable to log in after password reset
  Category: Authentication

#2 - TICK-011: SSO authentication broken after upgrade
  Category: Authentication

#3 - TICK-016: Two-factor authentication codes not working
  Category: Authentication


PART 3: Create Prompt Template with Anti-Hallucination Rules

In [15]:
# Define strict grounding prompt
# Prompt engineering is CRUCIAL for RAG - it tells the LLM how to use the context

# Key principles:
# 1. Be explicit about using ONLY the provided context
# 2. Define what to do when information is missing
# 3. Request citations for transparency and verification
# 4. Set the role/persona for appropriate tone

In [16]:
prompt_template = """You are SupportDesk AI, a technical support assistant that helps engineers troubleshoot issues using historical support ticket data.

CRITICAL RULES:
1. Answer using ONLY information from the provided context.
2. If the question is broad or underspecified, provide the best matching known issue(s) from context and state any assumptions.
3. If context is partially relevant, still provide the most likely troubleshooting guidance from relevant tickets.
4. If the answer is truly not present in context, say "I don't have enough information in the ticket history to answer that question."
5. DO NOT make up information or use external knowledge.
6. Always cite ticket IDs for every issue/resolution you mention.
7. If multiple tickets are relevant, summarize each briefly.

Context from support tickets:
{context}

Question: {question}

Helpful Answer (with ticket citations):"""

In [17]:
# Convert string template to ChatPromptTemplate
# This creates a reusable template with variable placeholders
PROMPT = ChatPromptTemplate.from_template(prompt_template)

print("✓ Prompt template created with anti-hallucination rules:")

✓ Prompt template created with anti-hallucination rules:


PART 4: Initialize LLM

In [18]:
if os.getenv("OPENAI_API_KEY"):
    print("✓ OpenAI API key found")

    # Initialize ChatOpenAI for generation
    llm = ChatOpenAI(
        model=os.getenv('OPENAI_CHAT_MODEL', 'gpt-4o-mini'),
        temperature=0,  # Temperature controls randomness (0 = deterministic, 2 = very creative)
        # For RAG, use temperature=0 to ensure consistent, factual responses
        timeout=120,  # Increase timeout for slower connections
        max_retries=3,  # Retry on transient failures
    )
    print(f"✓ Using {os.getenv('OPENAI_CHAT_MODEL', 'gpt-4o-mini')}")
else:
    print("⚠ OpenAI API key not found!")
    llm = None

✓ OpenAI API key found
✓ Using gpt-4o-mini


PART 5: Build RAG Chain using LCEL (LangChain Expression Language)

In [19]:
# Helper function to format retrieved documents
# This concatenates all retrieved document contents into a single context string
def format_docs(docs):
    """
    Convert a list of LangChain Document objects into a single context string.

    Why this helper exists:
    - Retrievers return `List[Document]` objects.
    - Prompt templates expect plain strings for `{context}`.
    - Joining with separators keeps boundaries visible to the LLM.
    """
    # Keep document boundaries explicit so the model can attribute facts by chunk.
    return "\n\n---\n\n".join([doc.page_content for doc in docs])

In [20]:
if llm:
    # Build RAG chain using LCEL (LangChain Expression Language)
    # LCEL allows you to chain components using the | operator (like Unix pipes)
    #
    # This chain does:
    # 1. Takes a question (string input)
    # 2. Retriever gets relevant docs, format_docs combines them
    # 3. PROMPT fills in {context} and {question} variables
    # 4. LLM generates answer based on filled prompt
    # 5. StrOutputParser extracts the string response
    #
    # The dict {"context": ..., "question": ...} creates the input for the prompt
    qa_chain = (
        {"context": retriever | format_docs, "question": RunnablePassthrough()}
        | PROMPT
        | llm
        | StrOutputParser()
    )
    print("✓ RAG chain assembled:")
    print("  Retriever → Context Injection → LLM → Answer")
    print("\nThis is the complete RAG pipeline! Query in → Answer out")
else:
    qa_chain = None
    print("⚠ LLM not available, showing architecture only")

✓ RAG chain assembled:
  Retriever → Context Injection → LLM → Answer

This is the complete RAG pipeline! Query in → Answer out


PART 6: Test the RAG System

In [21]:
test_queries = [
    "How do I fix authentication failures after password reset?",
    "What causes database connection timeouts?",
    "Why are emails not being delivered?",
    "How do I make the perfect pizza?"  # Should refuse to answer!
]

In [22]:
for query in test_queries:
    print("\n" + "="*80)
    print(f"QUERY: {query}")
    print("="*80)

    # Show retrieved context
    docs = retriever.invoke(query)
    print(f"\nRetrieved {len(docs)} relevant tickets:")
    for i, doc in enumerate(docs, 1):
        print(f"\n  [{i}] {doc.metadata['ticket_id']}: {doc.metadata['title']}")

    if qa_chain:
        # Generate answer
        print("\nGenerating answer...")
        result = qa_chain.invoke(query)

        print("\n" + "-"*80)
        print("ANSWER:")
        print("-"*80)
        print(result)

        print("\n" + "-"*80)
        print("SOURCE DOCUMENTS:")
        print("-"*80)
        for i, doc in enumerate(docs, 1):
            print(f"{i}. {doc.metadata['source']}")
    else:
        print("\n(LLM not configured - would generate answer here)")


QUERY: How do I fix authentication failures after password reset?

Retrieved 3 relevant tickets:

  [1] TICK-001: Users unable to log in after password reset

  [2] TICK-011: SSO authentication broken after upgrade

  [3] TICK-016: Two-factor authentication codes not working

Generating answer...

--------------------------------------------------------------------------------
ANSWER:
--------------------------------------------------------------------------------
To fix authentication failures after a password reset, you can refer to the resolution provided in Ticket ID: TICK-001. The issue was caused by an updated password hash algorithm while the session tokens remained active, leading to 'Invalid credentials' errors for users.

**Resolution Steps:**
1. Clear all active sessions for affected users.
2. Force re-authentication for those users.
3. Implement automatic session cleanup on password changes to prevent future occurrences.

By following these steps, you should be able to res

PART 7: Validation & Fallback

In [23]:
print("PART 7: Enhanced RAG with Answer Validation")

PART 7: Enhanced RAG with Answer Validation


In [24]:
def rag_with_validation(query, retriever, llm, min_similarity_score=0.5):
    """
        RAG pipeline with additional validation and fallback.

        Validation rule in this demo:
        - Use relevance score as a confidence proxy.
        - If the best document's relevance score is too low (< threshold),
            return a safe fallback instead of forcing an answer.

        Note:
        - similarity_search_with_relevance_scores() returns scores in [0, 1].
        - Higher = more similar (derived from cosine similarity).
        - This keeps things consistent with the cosine similarity concept
          taught in earlier modules (cosine similarity: -1 to 1).
        - This is a simple, practical guardrail for anti-hallucination behavior.
    """
    # Retrieve documents with relevance scores (0 = least relevant, 1 = most relevant)
    # Uses similarity_search_with_relevance_scores instead of similarity_search_with_score
    # because the latter returns raw cosine *distance* (0–2, lower=better), which is
    # confusing when we've been teaching cosine *similarity* (-1 to 1, higher=better).
    docs_with_scores = vector_store.similarity_search_with_relevance_scores(query, k=3)

    print(f"\nQuery: {query}")
    print(f"\nRelevance scores (cosine similarity: 0=no match, 1=identical):")
    for doc, score in docs_with_scores:
        print(f"  - {doc.metadata['ticket_id']}: {score:.4f}")

    # Use the best retrieved document as the confidence anchor.
    # If even the best match is weak, the whole answer should be treated as risky.
    best_score = docs_with_scores[0][1]

    # Relevance score: higher = more similar. Below 0.5 means the match is too weak to answer.
    if best_score < min_similarity_score:
        print(f"\n⚠ Best match relevance ({best_score:.4f}) is below threshold ({min_similarity_score}) — too dissimilar to answer confidently")
        return "I don't have enough relevant information in the ticket history to answer that question confidently."

    # If we pass the confidence gate, build context and ask the model normally.
    docs = [doc for doc, score in docs_with_scores]
    context = "\n\n---\n\n".join([doc.page_content for doc in docs])

    prompt = f"""{prompt_template.replace('{context}', context).replace('{question}', query)}"""

    if llm:
        # Use chat-model invocation directly and normalize return type to string.
        # `ChatOpenAI.invoke(...)` returns an AIMessage object in modern LangChain.
        response = llm.invoke(prompt)
        return response.content if hasattr(response, "content") else str(response)
    else:
        return "(LLM not configured)"

print("\nTesting validation logic:")
print("\n1. Relevant query (should answer):")


Testing validation logic:

1. Relevant query (should answer):


In [25]:
rag_with_validation(
    "How to fix database connection timeouts?",
    retriever,
    llm,
    min_similarity_score=0.5
)

print("\n2. Irrelevant query (should refuse):")
rag_with_validation(
    "What is the capital of France?",
    retriever,
    llm,
    min_similarity_score=0.5
)


Query: How to fix database connection timeouts?

Relevance scores (cosine similarity: 0=no match, 1=identical):
  - TICK-002: 0.5742
  - TICK-010: 0.4086
  - TICK-014: 0.4086

2. Irrelevant query (should refuse):

Query: What is the capital of France?

Relevance scores (cosine similarity: 0=no match, 1=identical):
  - TICK-003: 0.0285
  - TICK-015: 0.0165
  - TICK-016: 0.0146

⚠ Best match relevance (0.0285) is below threshold (0.5) — too dissimilar to answer confidently


"I don't have enough relevant information in the ticket history to answer that question confidently."

PART 8: Conversation with History (Multi-Turn RAG)

In [26]:
print("""
Problem with single-turn RAG:
  Turn 1: "How do I fix authentication failures?"  → good answer
  Turn 2: "How long did it take to resolve?"       → loses context! "it" = ???

Solution: Two-part fix:
  1. MessagesPlaceholder injects prior HumanMessage / AIMessage objects into the prompt
     so the LLM can understand references like "that issue" or "it".
  2. Query reformulation rewrites follow-up questions into standalone queries
     BEFORE retrieval, so the retriever searches for the right documents.
     e.g. "How do I fix it?" + history → "How do I fix authentication failures?"
""")


Problem with single-turn RAG:
  Turn 1: "How do I fix authentication failures?"  → good answer
  Turn 2: "How long did it take to resolve?"       → loses context! "it" = ???

Solution: Two-part fix:
  1. MessagesPlaceholder injects prior HumanMessage / AIMessage objects into the prompt
     so the LLM can understand references like "that issue" or "it".
  2. Query reformulation rewrites follow-up questions into standalone queries
     BEFORE retrieval, so the retriever searches for the right documents.
     e.g. "How do I fix it?" + history → "How do I fix authentication failures?"



In [27]:
if llm:
    # ── Step 1: Query Reformulation ────────────────────────────────────
    # The retriever only sees the raw question string.
    # "What was the resolution for that ticket?" → retriever gets vague query.
    # Fix: Use the LLM to rewrite follow-ups into standalone queries first.
    condense_prompt = ChatPromptTemplate.from_messages([
        ("system",
         "Given the chat history and a follow-up question, rephrase the "
         "follow-up as a standalone question that includes all necessary "
         "context from the history. If the question is already standalone, "
         "return it unchanged."),
        MessagesPlaceholder(variable_name="chat_history"),
        ("human", "{question}"),
    ])

    # Chain that rewrites the question: dict → standalone question string
    condense_chain = condense_prompt | llm | StrOutputParser()

    # ── Step 2: Conversation-aware prompt ──────────────────────────────
    # MessagesPlaceholder expands the history list into the prompt at call time
    conv_prompt = ChatPromptTemplate.from_messages([
        ("system", """You are SupportDesk AI. Answer using the ticket context below and the chat history.
    Use chat history to resolve references like "that issue" or "that ticket".
    For factual claims, prioritize the retrieved context.
    If information is not available in context or history, say "I don't have that information."
    Always cite ticket IDs when available.

Context:
{context}"""),
        MessagesPlaceholder(variable_name="chat_history"),
        ("human", "{question}"),
    ])

    def ask_with_history(question, history):
        """Ask a question with query reformulation and history tracking.

        On the first turn (empty history), skip the condense step and send
        the question directly to the retriever. On follow-up turns, rewrite
        the question into a standalone query so the retriever finds the
        right documents (e.g. "that ticket" → "TICK-001").
        """
        if not history:
            # First turn: no history to condense, retrieve directly
            standalone = question
        else:
            # Follow-up turn: rewrite using history context
            # e.g. "What was the resolution for that ticket?" →
            #      "What was the resolution for ticket TICK-001?"
            standalone = condense_chain.invoke({
                "question": question, "chat_history": history
            })

        # Retrieve docs using the standalone query and generate the answer
        context = format_docs(retriever.invoke(standalone))
        answer = (conv_prompt | llm | StrOutputParser()).invoke({
            "context": context,
            "chat_history": history,
            "question": question,  # Original question, not the rewritten one
        })

        history.append(HumanMessage(content=question))
        history.append(AIMessage(content=answer))
        return answer

    # ── Demonstrate a 3-turn conversation ──────────────────────────────────
    print("Multi-turn conversation demo:\n")
    history = []

    q1 = "How do I fix authentication failures after a password reset?"
    print(f"Turn 1 — User: {q1}")
    a1 = ask_with_history(q1, history)
    print(f"         Assistant: {a1[:300]}{'...' if len(a1) > 300 else ''}")
    print(f"         [chat_history now has {len(history)} messages]")

    q2 = "What was the ticket ID for that issue?"
    print(f"\nTurn 2 — User: {q2}")
    a2 = ask_with_history(q2, history)
    print(f"         Assistant: {a2[:300]}{'...' if len(a2) > 300 else ''}")
    print(f"         [chat_history now has {len(history)} messages]")

    q3 = "What was the resolution for that ticket?"
    print(f"\nTurn 3 — User: {q3}")
    a3 = ask_with_history(q3, history)
    print(f"         Assistant: {a3[:300]}{'...' if len(a3) > 300 else ''}")
    print(f"         [chat_history now has {len(history)} messages]")

    print(f"\n✓ History contains {len(history)} messages ({len(history)//2} complete turns)")
    print("TIP: In production, cap history to avoid token bloat:")
    print("       history = history[-6:]  # keep last 3 turns")

else:
    print("(LLM not configured — would run multi-turn conversation here)")

Multi-turn conversation demo:

Turn 1 — User: How do I fix authentication failures after a password reset?
         Assistant: To fix authentication failures after a password reset, you can follow the resolution steps from Ticket ID TICK-001. The issue was caused by an updated password hash algorithm without invalidating session tokens. Here’s what you can do:

1. Clear all active sessions for the affected users.
2. Force r...
         [chat_history now has 2 messages]

Turn 2 — User: What was the ticket ID for that issue?
         Assistant: The ticket ID for that issue is TICK-001.
         [chat_history now has 4 messages]

Turn 3 — User: What was the resolution for that ticket?
         Assistant: The resolution for Ticket ID TICK-001 was as follows:

1. Cleared all active sessions for affected users.
2. Forced re-authentication for those users.
3. Implemented automatic session cleanup on password changes to prevent similar issues in the future.

This addressed the authentication f

PART 9: Interactive Demo

In [28]:
if qa_chain:
    print("\nSupportDesk RAG Assistant Ready!")
    print("Ask questions about support ticket history.")
    print("Type 'quit' to exit.\n")

    while True:
        user_query = input("You: ").strip()

        if user_query.lower() in ['quit', 'exit', 'q']:
            print("Goodbye!")
            break

        if not user_query:
            continue

        print("\nAssistant: ", end="")
        answer = qa_chain.invoke(user_query)
        print(answer)

        docs = retriever.invoke(user_query)
        print(f"\n📎 Sources: {', '.join([doc.metadata['ticket_id'] for doc in docs])}")
        print()
else:
    print("\n⚠ Interactive mode requires OpenAI API key")
    print("Set OPENAI_API_KEY to try the interactive assistant!")


SupportDesk RAG Assistant Ready!
Ask questions about support ticket history.
Type 'quit' to exit.

You: tick 001

Assistant: I don't have enough information in the ticket history to answer that question.

📎 Sources: TICK-016, TICK-014, TICK-003

You: password not working after reset

Assistant: The issue of passwords not working after a reset is addressed in Ticket ID: TICK-001. 

**Summary of the Issue:**
- **Problem:** Multiple users reported authentication failures after performing a password reset, receiving an error message stating 'Invalid credentials'. This issue began following a recent security patch deployment.
- **Resolution:** It was discovered that the password hash algorithm had been updated, but the session tokens were not invalidated. The solution involved clearing all active sessions and forcing re-authentication. Additionally, automatic session cleanup was implemented upon password changes.

If you are experiencing similar issues, consider checking if session tokens 